# 旧期货湖迁移实时状态

这是只读监控 Notebook：不写湖仓、不继续或中断迁移。

判断“正常运行”不能只看一个信号，至少同时看：

- 迁移 PID 仍存活；
- 采样期间 CPU 时间增长，或 staging 文件数/最新修改时间前进；
- `events.jsonl` 没有新的失败或回滚事件；
- 正式湖没有意外遗留的 staging/backup/failed 目录。

CPU 暂时不变可能是磁盘 I/O；单次采样不足以宣告停滞，应隔几分钟重跑最后一个单元格。

In [ ]:
from __future__ import annotations

import json
import pathlib
import sys
import time
from datetime import datetime, timezone

import psutil


# 使用项目约定的标记文件定位根目录。
project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()  # 当前工作目录

for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        PROJECT_ROOT = candidate_root
        break
else:
    raise RuntimeError("未找到项目根目录")

from config.settings import settings


FORMAL_LAKE_ROOT = settings.futures_lake_root
FORMAL_SILVER = FORMAL_LAKE_ROOT / "silver"
REPORT_ROOT = PROJECT_ROOT / "00_draft_collection_02" / "migration_reports"
TRANSACTION_PARENT = FORMAL_LAKE_ROOT.parent

transaction_roots = sorted(
    TRANSACTION_PARENT.glob(".legacy-migration-*"),
    key=lambda path: path.stat().st_mtime_ns,
)
report_paths = sorted(
    REPORT_ROOT.glob("legacy-futures-lake-*.json"),
    key=lambda path: path.stat().st_mtime_ns,
)
latest_transaction = transaction_roots[-1] if transaction_roots else None
latest_report = report_paths[-1] if report_paths else None
report_is_newer = (
    latest_report is not None
    and (
        latest_transaction is None
        or latest_report.stat().st_mtime_ns >= latest_transaction.stat().st_mtime_ns
    )
)
if latest_transaction is not None and not report_is_newer:
    TRANSACTION_ROOT = latest_transaction
    RUN_ID = TRANSACTION_ROOT.name.removeprefix(".legacy-migration-")
    EVENT_PATH = TRANSACTION_ROOT / "events.jsonl"
    CANDIDATE_SILVER = TRANSACTION_ROOT / "candidate_lake" / "silver"
    REPORT_PATH = REPORT_ROOT / f"legacy-futures-lake-{RUN_ID}.json"
elif latest_report is not None:
    # 成功后事务目录会删除，此时从最新持久报告判断。
    TRANSACTION_ROOT = None
    REPORT_PATH = latest_report
    RUN_ID = REPORT_PATH.stem.removeprefix("legacy-futures-lake-")
    EVENT_PATH = None
    CANDIDATE_SILVER = None
else:
    raise FileNotFoundError("未发现迁移事务目录或持久报告")

print(f"检查时间：{datetime.now().astimezone().isoformat(timespec='seconds')}")
print(f"事务目录：{TRANSACTION_ROOT or '已清理'}")
print(f"正式湖根：{FORMAL_LAKE_ROOT}")

In [ ]:
# 严格读取事务事件；不忽略损坏的 JSON 行。
event_bytes = EVENT_PATH.read_bytes() if EVENT_PATH and EVENT_PATH.is_file() else b""
if event_bytes and not event_bytes.endswith(b"\n"):
    raise ValueError("events.jsonl 尾部不完整")

events = [json.loads(line) for line in event_bytes.splitlines()]
actions = [str(event.get("action")) for event in events]
latest_events = events[-8:]

print(f"事件数：{len(events):,}")
print("最近事件：")
for event in latest_events:
    print("  ", json.dumps(event, ensure_ascii=False))

print("\n关键阶段计数：")
for action in [
    "c07_completed",
    "c08_completed",
    "candidate_fully_validated",
    "formal_to_backup",
    "candidate_to_formal",
    "formal_reread_completed",
    "old_lake_and_backups_removed",
]:
    print(f"  {action}: {actions.count(action)}")

In [ ]:
# 查找命令行中明确带有迁移脚本或当前事务目录的 Python 进程。
def migration_processes() -> dict[int, dict[str, object]]:
    matched: dict[int, dict[str, object]] = {}
    for process in psutil.process_iter(["pid", "name", "cmdline", "create_time"]):
        try:
            command = " ".join(process.info.get("cmdline") or [])
            transaction_match = (
                TRANSACTION_ROOT is not None and str(TRANSACTION_ROOT) in command
            )
            if (
                "migrate_pre_rebuild_futures_lake.py" not in command
                and not transaction_match
                and "b08_full_minute_quality.py" not in command
            ):
                continue
            matched[process.pid] = {
                "pid": process.pid,
                "name": process.info.get("name"),
                "command": command,
                "cpu_seconds": process.cpu_times().user + process.cpu_times().system,
                "working_set_mb": process.memory_info().rss / 1024**2,
                "started_at": datetime.fromtimestamp(
                    process.info["create_time"], timezone.utc
                ).astimezone().isoformat(timespec="seconds"),
            }
        except (psutil.AccessDenied, psutil.NoSuchProcess, psutil.ZombieProcess):
            continue
    return matched


before = migration_processes()
time.sleep(10)
after = migration_processes()

print(f"匹配的运行进程：{len(after)}")
for pid, current in after.items():
    previous_cpu = float(before.get(pid, {}).get("cpu_seconds", current["cpu_seconds"]))
    cpu_delta = float(current["cpu_seconds"]) - previous_cpu
    print(
        f"  PID={pid}; CPU增量={cpu_delta:.2f}s/10s; "
        f"内存={current['working_set_mb']:.0f}MB; 启动={current['started_at']}"
    )

CPU_IS_ADVANCING = any(
    pid in before
    and float(current["cpu_seconds"]) - float(before[pid]["cpu_seconds"]) > 0.2
    for pid, current in after.items()
)

In [ ]:
# 统计 c08 临时目录，并找出最新写入的 Parquet。
staging_roots = (
    sorted(
        path
        for path in CANDIDATE_SILVER.glob(".c08-*")
        if path.is_dir()
    )
    if CANDIDATE_SILVER and CANDIDATE_SILVER.is_dir()
    else []
)

latest_staging_mtime = None
for staging_root in staging_roots:
    parquet_files = list(staging_root.rglob("*.parquet"))
    latest_file = max(parquet_files, key=lambda path: path.stat().st_mtime_ns, default=None)
    print(f"{staging_root.name}: parquet={len(parquet_files):,}")
    if latest_file is not None:
        modified = datetime.fromtimestamp(latest_file.stat().st_mtime).astimezone()
        latest_staging_mtime = max(
            latest_staging_mtime or modified,
            modified,
        )
        print(f"  最新文件：{latest_file.relative_to(staging_root)}")
        print(f"  修改时间：{modified.isoformat(timespec='seconds')}")

if not staging_roots:
    print("当前没有 c08 staging 目录。")

formal_recovery = sorted(
    path.name
    for path in FORMAL_SILVER.iterdir()
    if path.name.startswith(".")
    and any(token in path.name for token in ("staging", "backup", "failed", "pre-migration"))
)
print(f"正式 silver 恢复目录：{formal_recovery or '无'}")

In [ ]:
# 结合进程、事件、报告和正式恢复目录给出当前时刻的判断。
report = None
if REPORT_PATH.is_file():
    report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))

if after:
    if CPU_IS_ADVANCING:
        verdict = "正常运行：迁移进程存活，且 10 秒内 CPU 时间在增长。"
    else:
        verdict = (
            "进程存活，但本次 CPU 采样未前进。"
            "这可能是 I/O 等待；隔 2–5 分钟重跑本 Notebook，不要仅凭一次采样中断任务。"
        )
elif report and report.get("completed_at"):
    verdict = "迁移已完成，当前不应再有迁移进程。"
elif report and report.get("failed_at"):
    rollback_errors = report.get("rollback_errors") or []
    verdict = (
        "迁移已在错误后停止。"
        + (
            "报告显示回滚无二次错误；应先诊断报告再使用阶段化恢复入口。"
            if not rollback_errors
            else f"回滚仍有错误：{rollback_errors}。不得自动续跑。"
        )
    )
else:
    verdict = "未找到运行进程或明确完成/失败报告；需要手工审计事务现场。"

print("\n=== 当前判断 ===")
print(verdict)
if report and report.get("error"):
    print(f"最近错误：{report['error']}")
if formal_recovery:
    print("警告：正式 silver 存在事务恢复目录，必须人工审计后再续跑。")